# Intraday lab — backtest a day-trading rule, then try to break it

Day trading is its own project here (`intraday/`), separate from the swing
research in `rhengine/`. No rule has been chosen yet; the ORB below is only a
worked example. Workflow (`intraday/docs/GUIDE.md`):

1. **Log the idea** in intraday's own ledger before running it.
2. Develop on bars **before** `HOLDOUT_START` only.
3. `evaluate()` runs every check: look-ahead, sample size, multiple testing,
   deflated Sharpe, random entries, 2x slippage, sub-periods, survivorship,
   walk-forward, plateau.
4. Only when nothing FAILs: run `holdout()` **once**, then record the outcome.

`CLEARS` is permission to paper-trade, not to trade. There is no order code in
`intraday/`. Costs assume IBKR (`intraday/instruments.py`).

In [ ]:
import os, sys, tempfile
from pathlib import Path
root = Path.cwd()
while not (root / "intraday").is_dir():
    root = root.parent
sys.path.insert(0, str(root)); os.chdir(root)

import pandas as pd
from rhengine import trials as T          # ledger maths only
from intraday import data, engine, instruments as I, strategies as St, validate as V

## Data

Put a one-minute export at `intraday/data/MES_1m.csv` (Databento `ohlcv-1m`,
parent symbol `MES.FUT`, or any CSV with `ts, open, high, low, close, volume`).
Without it the lab runs on a synthetic random walk and logs to a throwaway
ledger, so nothing it prints means anything.

In [ ]:
DATA = root / "intraday" / "data" / "MES_1m.csv"
HOLDOUT_START = "2025-07-01"          # never develop on bars at or after this

if DATA.exists():
    raw = data.rth(data.load_bars(str(DATA)))
    bars = data.continuous(raw) if "contract" in raw.columns else raw
    LEDGER = V.LEDGER                   # intraday/data/trials.jsonl
    LOOKS = V.HOLDOUT_LOG
else:
    bars = data.synthetic_bars(500, seed=0)
    HOLDOUT_START = str(bars.index[int(len(bars) * 0.8)].date())
    _tmp = tempfile.mkdtemp()
    LEDGER = os.path.join(_tmp, "trials.jsonl")
    LOOKS = os.path.join(_tmp, "holdout.jsonl")
    print("NO DATA FILE: synthetic random walk + throwaway ledger. Every number is luck.")

dev = bars[bars.index < pd.Timestamp(HOLDOUT_START, tz=data.TZ)]
print(f"{len(dev):,} dev bars, {dev.index[0]} -> {dev.index[-1]}; holdout from {HOLDOUT_START}")

## 1. Log the idea before you run it

One label per idea. Changing the idea after seeing a number is a new trial.

In [ ]:
TRIAL = "example: orb 15m, stop at far side of range, MES"   # replace with your idea
if not V.is_logged(TRIAL, ledger=LEDGER):
    T.append(T.Trial(family=V.FAMILY, label=TRIAL, source="intraday/docs/IDEAS.md"), LEDGER)
print(T.count(V.FAMILY, path=LEDGER), "trials logged in family", V.FAMILY)

## 2. One backtest — look at the trades, not just the total

In [ ]:
strategy = St.OpeningRangeBreakout(or_minutes=15)
engine.causality_check(strategy, dev)          # raises if a signal reads the future
res = engine.backtest(strategy, dev, I.MES, trial=TRIAL)
print(res.summary())
res.trades[["entry_ts", "exit_ts", "side", "entry_px", "exit_px", "net", "reason"]].tail(10)

## 3. Every check

`GRID` is the full set of settings you are choosing between. It is charged to
the ledger (one row, n = combos - 1) before the burden is computed.

In [ ]:
GRID = {"or_minutes": [5, 15, 30], "take_r": [None, 1.0, 2.0]}
rep = V.evaluate(strategy, dev, I.MES, trial=TRIAL, survivorship="continuous_futures",
                 grid=GRID, ledger=LEDGER, holdout_start=HOLDOUT_START)
V.print_report(rep)

In [ ]:
rep["walk_forward"].folds if rep["walk_forward"] is not None else "no walk-forward"

In [ ]:
try:
    ax = rep["result"].daily.cumsum().plot(title="cumulative net USD (dev)", figsize=(10, 3))
    rep["sweep"].daily.cumsum().plot(ax=ax, alpha=0.2, legend=False, color="grey")
except ImportError:
    print("matplotlib not installed; skipping the chart")

## 4. Holdout — once, last, only if nothing above FAILED

`holdout()` writes the look to `intraday/data/holdout.jsonl` before it runs.
Uncomment and run one time.

In [ ]:
# h = V.holdout(strategy, bars, I.MES, trial=TRIAL, holdout_start=HOLDOUT_START,
#               ledger=LEDGER, look_log=LOOKS)
# rep = V.evaluate(strategy, dev, I.MES, trial=TRIAL, survivorship="continuous_futures",
#                  grid=GRID, ledger=LEDGER, holdout_start=HOLDOUT_START, holdout_result=h)
# V.print_report(rep)

## 5. Record the outcome — rejections count as much as results

```
python -m intraday log --label outcome --resolves "<your idea>" --outcome rejected
```